# Image Denoising with Improved U-Net - Training Notebook

This notebook trains the Improved U-Net model on a **free T4 GPU**.

**Instructions:**
1. Go to **Runtime > Change runtime type > GPU (T4)**
2. Run all cells in order
3. Download the trained `best_model.pth` at the end

Estimated training time: **~1-2 hours** (vs days on CPU)

## Step 1: Clone the repository

In [ ]:
!git clone https://github.com/govind2345/Image-Denoising-with-Improved-U-Net-Architecture.git
%cd Image-Denoising-with-Improved-U-Net-Architecture

## Step 2: Upload & Extract the MVTec Dataset

Upload the `mvtec_anomaly_detection.tar.xz` file when prompted.

In [ ]:
from google.colab import files
import os

# Option A: Upload from your computer
print("Upload mvtec_anomaly_detection.tar.xz (or skip if using Google Drive)")
try:
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    print(f"Uploaded: {filename}")
except:
    print("Upload skipped. Use Option B below if you have the file on Google Drive.")

In [ ]:
# Option B: Copy from Google Drive (uncomment and modify the path)
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/mvtec_anomaly_detection.tar.xz .

In [ ]:
# Extract the dataset
!tar -xf mvtec_anomaly_detection.tar.xz
print("Dataset extracted!")

## Step 3: Create the Denoising Dataset folder structure

In [ ]:
import os, shutil, random, glob
from PIL import Image
import numpy as np

SRC = '.'  # extracted MVTec categories are in current dir
DST = 'Denoising_Dataset_prepared'

CATEGORIES = [
    'bottle', 'cable', 'capsule', 'carpet', 'grid',
    'hazelnut', 'leather', 'metal_nut', 'pill', 'screw',
    'tile', 'toothbrush', 'transistor', 'wood', 'zipper'
]

random.seed(42)

def add_noise(img_array, noise_level=25):
    noise = np.random.normal(0, noise_level, img_array.shape).astype(np.float32)
    noisy = np.clip(img_array.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    return noisy

for cat in CATEGORIES:
    good_dir = os.path.join(SRC, cat, 'train', 'good')
    if not os.path.isdir(good_dir):
        print(f"Skipping {cat}: no train/good folder")
        continue

    images = sorted(glob.glob(os.path.join(good_dir, '*.png')))
    random.shuffle(images)

    split = int(0.8 * len(images))
    splits = {'Train': images[:split], 'Val': images[split:]}

    for mode, img_list in splits.items():
        deg_dir = os.path.join(DST, cat, mode, 'Degraded_image')
        gt_dir  = os.path.join(DST, cat, mode, 'GT_clean_image')
        mk_dir  = os.path.join(DST, cat, mode, 'Defect_mask')
        os.makedirs(deg_dir, exist_ok=True)
        os.makedirs(gt_dir, exist_ok=True)
        os.makedirs(mk_dir, exist_ok=True)

        for img_path in img_list:
            fname = os.path.basename(img_path)
            stem  = os.path.splitext(fname)[0]

            img = Image.open(img_path).convert('RGB')
            arr = np.array(img)
            noisy = add_noise(arr)

            Image.fromarray(noisy).save(os.path.join(deg_dir, fname))
            img.save(os.path.join(gt_dir, fname))

            mask = Image.new('L', img.size, 0)
            mask.save(os.path.join(mk_dir, f'{stem}_mask.png'))

    print(f"{cat}: {len(splits['Train'])} train / {len(splits['Val'])} val")

print("\nDataset preparation complete!")

## Step 4: Verify GPU is available

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU! Go to Runtime > Change runtime type > GPU")

## Step 5: Train the Model (Full Training)

This trains at **256x256 resolution** for **50 epochs** with early stopping.
Should take about **1-2 hours on T4 GPU**.

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from tqdm.notebook import tqdm
import numpy as np
import os
import sys

# Add current dir to path and import model components
sys.path.insert(0, '.')
from model import (
    DenoisingDataset, ImprovedUNet, PriorityFocusedLoss,
    evaluate_model, save_metrics, plot_training_curves, calculate_model_size
)

# --- Config ---
NUM_EPOCHS = 50
BATCH_SIZE = 8
IMG_SIZE = 256
LR = 1e-4
PATIENCE = 15

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on: {device}")

# --- Data ---
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

train_dataset = DenoisingDataset('Denoising_Dataset_prepared', transform, 'Train')
val_dataset   = DenoisingDataset('Denoising_Dataset_prepared', transform, 'Val')

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_dataset)} images, {len(train_loader)} batches")
print(f"Val:   {len(val_dataset)} images, {len(val_loader)} batches")

# --- Model ---
model = ImprovedUNet(skip_connections=[False, True, True, True]).to(device)
info = calculate_model_size(model)
print(f"Parameters: {info['parameters']:,} | Size: {info['size_mb']:.1f} MB")

# --- Training Setup ---
criterion = PriorityFocusedLoss().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2, eta_min=1e-7)

best_metrics = {'defect_ssim': 0, 'epoch': 0}
history = {'train_loss': [], 'val_metrics': []}
no_improve = 0

# --- Training Loop ---
for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss = 0

    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS}')
    for batch in pbar:
        degraded, clean, mask = [x.to(device) for x in batch]
        if mask.shape[1] == 3:
            mask = mask.mean(dim=1, keepdim=True)

        optimizer.zero_grad()
        outputs = model(degraded, mask)
        loss = criterion(outputs, clean, mask)

        if torch.isnan(loss):
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_loss = train_loss / len(train_loader)

    # Validation
    model.eval()
    val_metrics = evaluate_model(model, val_loader, device)
    scheduler.step(epoch)

    defect_ssim = val_metrics.get('defect_ssim', 0)
    if np.isnan(defect_ssim):
        defect_ssim = 0

    # Save best
    if defect_ssim > best_metrics['defect_ssim']:
        best_metrics = {
            'defect_ssim': defect_ssim,
            'epoch': epoch,
            'state_dict': model.state_dict()
        }
        torch.save(best_metrics, 'best_model.pth')
        no_improve = 0
        marker = ' << BEST'
    else:
        no_improve += 1
        marker = ''

    history['train_loss'].append(avg_loss)
    history['val_metrics'].append(val_metrics)

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Loss: {avg_loss:.4f} | "
          f"PSNR: {val_metrics.get('psnr', 0):.2f} | SSIM: {val_metrics.get('ssim', 0):.4f} | "
          f"Defect SSIM: {defect_ssim:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}{marker}")

    # Early stopping
    if no_improve >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1} (no improvement for {PATIENCE} epochs)")
        break

print(f"\nTraining complete! Best epoch: {best_metrics['epoch']+1}, Best Defect SSIM: {best_metrics['defect_ssim']:.4f}")

## Step 6: Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history['train_loss'], 'b-', linewidth=2)
axes[0].set_title('Training Loss', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)

# PSNR
psnr_vals = [m.get('psnr', 0) for m in history['val_metrics']]
axes[1].plot(psnr_vals, 'g-', linewidth=2)
axes[1].set_title('Validation PSNR', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('PSNR (dB)')
axes[1].grid(True, alpha=0.3)

# SSIM
ssim_vals = [m.get('ssim', 0) for m in history['val_metrics']]
axes[2].plot(ssim_vals, 'r-', linewidth=2)
axes[2].set_title('Validation SSIM', fontsize=14)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('SSIM')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves_gpu.png', dpi=150)
plt.show()
print('Saved training_curves_gpu.png')

## Step 7: Test the Model

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import torchvision.transforms as transforms

# Load best model
checkpoint = torch.load('best_model.pth', weights_only=False)
model.load_state_dict(checkpoint['state_dict'])
model.eval()

# Get sample images
fig, axes = plt.subplots(3, 3, figsize=(15, 15))
titles = ['Noisy Input', 'Model Output', 'Ground Truth']

for i in range(3):
    idx = i * (len(val_dataset) // 3)
    degraded, clean, mask = val_dataset[idx]

    degraded_t = degraded.unsqueeze(0).to(device)
    mask_t = mask.unsqueeze(0).to(device)
    if mask_t.shape[1] == 3:
        mask_t = mask_t.mean(dim=1, keepdim=True)

    with torch.no_grad():
        output = model(degraded_t, mask_t)

    imgs = [
        degraded.numpy().transpose(1, 2, 0),
        np.clip(output[0].cpu().numpy().transpose(1, 2, 0), 0, 1),
        clean.numpy().transpose(1, 2, 0),
    ]

    for j, (img, title) in enumerate(zip(imgs, titles)):
        axes[i][j].imshow(img)
        axes[i][j].set_title(title, fontsize=13)
        axes[i][j].axis('off')

plt.suptitle('Denoising Results (GPU-Trained Model)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('sample_results_gpu.png', dpi=150)
plt.show()

## Step 8: Download the Trained Model

Download `best_model.pth` and place it in your project folder.
Then run `python app.py` locally to use the improved model.

In [ ]:
from google.colab import files

print(f"Model file size: {os.path.getsize('best_model.pth') / 1e6:.1f} MB")
print(f"Best epoch: {checkpoint.get('epoch', '?') + 1}")
print(f"Best Defect SSIM: {checkpoint.get('defect_ssim', '?')}")
print("\nDownloading best_model.pth...")
files.download('best_model.pth')